# Trazar el agente

El [capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/observabilidad.html) empieza con la frase que justifica todo lo demás: un sistema generativo **no lanza excepciones**. Devuelve algo correcto en forma y equivocado en contenido, con código 200 y en el tiempo esperado.

Los cuadernos anteriores lo enseñaron sin querer varias veces. El agente contestó que la nota de cálculo era un 8.0 cuando el 8.0 era de otra asignatura. El clasificador mandó una pregunta de normativa a la ruta del expediente y soltó datos personales. Ninguna de las dos cosas lanzó nada: las dos parecían respuestas.

Este cuaderno instrumenta el agente de la secretaría para que esos fallos dejen rastro.

## Sin plataforma

No vamos a levantar ningún servicio. Usamos **OpenTelemetry** con las convenciones semánticas para IA generativa, que es lo que el capítulo recomienda exigir, y un exportador en memoria para poder mirar los tramos aquí mismo.

Esa decisión no es solo comodidad para un cuaderno. Es exactamente la propiedad que hace que un sistema de observabilidad no os secuestre: si emitís en el formato estándar, cambiar de herramienta de análisis es cambiar el exportador y nada más.

## Preparación

In [ ]:
!pip install -q opentelemetry-sdk opentelemetry-semantic-conventions duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()
con = ctx.conectar()

## El recolector

Tres piezas: un exportador que guarda los tramos en memoria, un proveedor que se los pasa y un `tracer` que es lo que usaremos para abrir tramos.

En producción, lo único que cambia es el exportador: se sustituye por uno OTLP apuntando a donde sea. El código instrumentado no se toca.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

memoria = InMemorySpanExporter()
proveedor = TracerProvider()
proveedor.add_span_processor(SimpleSpanProcessor(memoria))
trace.set_tracer_provider(proveedor)

tracer = trace.get_tracer("secretaria")
print("Recolector listo.")

## Los nombres los pone el estándar

Aquí está lo que ahorra el trabajo. No hay que decidir cómo llamar a cada atributo: las convenciones semánticas de IA generativa ya fijan los nombres, y vienen en un paquete.

In [ ]:
from opentelemetry.semconv._incubating.attributes import gen_ai_attributes as ga
from opentelemetry.semconv._incubating.attributes.gen_ai_attributes import (
    GenAiOperationNameValues as Operacion,
)

print("Tipos de operación que contempla el estándar:")
for op in Operacion:
    print(f"  {op.value}")

print("\nLa lista del capítulo, con su nombre estándar:")
for que, atributo in [
    ("el contexto exacto", ga.GEN_AI_INPUT_MESSAGES),
    ("las instrucciones", ga.GEN_AI_SYSTEM_INSTRUCTIONS),
    ("la respuesta", ga.GEN_AI_OUTPUT_MESSAGES),
    ("modelo pedido", ga.GEN_AI_REQUEST_MODEL),
    ("modelo que respondió", ga.GEN_AI_RESPONSE_MODEL),
    ("temperatura", ga.GEN_AI_REQUEST_TEMPERATURE),
    ("tokens de entrada", ga.GEN_AI_USAGE_INPUT_TOKENS),
    ("tokens de salida", ga.GEN_AI_USAGE_OUTPUT_TOKENS),
    ("tokens de caché", ga.GEN_AI_USAGE_CACHE_READ_INPUT_TOKENS),
    ("tokens de razonamiento", ga.GEN_AI_USAGE_REASONING_OUTPUT_TOKENS),
    ("versión del prompt", ga.GEN_AI_PROMPT_NAME),
    ("herramienta llamada", ga.GEN_AI_TOOL_NAME),
    ("argumentos", ga.GEN_AI_TOOL_CALL_ARGUMENTS),
    ("resultado", ga.GEN_AI_TOOL_CALL_RESULT),
]:
    print(f"  {que:26s} {atributo}")

Compárese con [la lista del capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/observabilidad.html#la-unidad-de-observación-es-la-traza). Está entera, salvo la latencia, y la latencia no necesita atributo porque **un tramo ya sabe cuándo empezó y cuándo acabó**. Es el primer regalo de usar trazas en lugar de registros sueltos.

Los identificadores de usuario y de sesión sí hay que ponerlos, y ahí el estándar no manda: se usan atributos propios con un prefijo del dominio.

## El agente instrumentado

Ahora el agente de siempre, con tres tramos anidados que siguen la estructura que dibuja el capítulo: uno por la petición entera (`invoke_agent`), uno por cada llamada al modelo (`chat`) y uno por cada ejecución de herramienta (`execute_tool`).

In [ ]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)
modelo.eval()

PATRON = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)
SISTEMA = ("Eres el asistente de la secretaría académica. Para responder debes llamar "
           "a una de las herramientas disponibles. No respondas de memoria.")
VERSION_PROMPT = "secretaria-sistema-v3"
ALUMNO = "A2023001"


def consultar_plazo(tramite: str) -> str:
    filas = con.execute("""
        select tramite, fecha_inicio, fecha_fin from dim_plazo
        where tramite ilike '%' || ? || '%' order by fecha_inicio limit 3
    """, [tramite]).fetchall()
    if not filas:
        return f"No existe el trámite '{tramite}'."
    return "; ".join(f"{t}: del {i} al {f}" for t, i, f in filas)


def consultar_expediente(asignatura: str = "") -> str:
    filas = con.execute("""
        select s.asignatura, c.nota
        from fct_matriculas m
        join dim_asignatura s on s.asignatura_id = m.asignatura_id
        left join fct_calificaciones c on c.matricula_id = m.matricula_id
        where m.alumno_id = ? and (? = '' or lower(s.asignatura) like '%' || lower(?) || '%')
        order by s.asignatura limit 6
    """, [ALUMNO, asignatura, asignatura]).fetchall()
    if not filas:
        return "No estás matriculado de eso."
    return "; ".join(f"{a}: {n if n is not None else 'sin nota'}" for a, n in filas)


CATALOGO = {"consultar_plazo": consultar_plazo,
            "consultar_expediente": consultar_expediente}

ESQUEMAS = [
    {"type": "function", "function": {
        "name": "consultar_plazo",
        "description": "Fechas de inicio y fin de un trámite administrativo.",
        "parameters": {"type": "object", "properties": {
            "tramite": {"type": "string", "description": "beca, matricula, tfg, revision"}},
            "required": ["tramite"]}}},
    {"type": "function", "function": {
        "name": "consultar_expediente",
        "description": "Asignaturas y notas del alumno que pregunta.",
        "parameters": {"type": "object", "properties": {
            "asignatura": {"type": "string"}}, "required": []}}},
]

In [ ]:
def llamar_modelo(mensajes):
    with tracer.start_as_current_span("chat") as tramo:
        texto = tok.apply_chat_template(mensajes, tools=ESQUEMAS, tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)
        entrada = tok(texto, return_tensors="pt")
        with torch.no_grad():
            salida = modelo.generate(**entrada, max_new_tokens=100, do_sample=False,
                                     pad_token_id=tok.eos_token_id)
        nuevos = salida[0][entrada.input_ids.shape[1]:]
        respuesta = tok.decode(nuevos, skip_special_tokens=True).strip()

        n_entrada, n_salida = int(entrada.input_ids.shape[1]), int(len(nuevos))
        tramo.set_attribute(ga.GEN_AI_OPERATION_NAME, Operacion.CHAT.value)
        tramo.set_attribute(ga.GEN_AI_PROVIDER_NAME, "transformers")
        tramo.set_attribute(ga.GEN_AI_REQUEST_MODEL, ctx.modelo)
        tramo.set_attribute(ga.GEN_AI_RESPONSE_MODEL, ctx.modelo)
        tramo.set_attribute(ga.GEN_AI_REQUEST_TEMPERATURE, 0.0)
        tramo.set_attribute(ga.GEN_AI_PROMPT_NAME, VERSION_PROMPT)
        tramo.set_attribute(ga.GEN_AI_USAGE_INPUT_TOKENS, n_entrada)
        tramo.set_attribute(ga.GEN_AI_USAGE_OUTPUT_TOKENS, n_salida)
        # El contexto EXACTO, no la plantilla. Es lo que explica el 90% de las rarezas.
        tramo.set_attribute(ga.GEN_AI_INPUT_MESSAGES, json.dumps(mensajes, ensure_ascii=False))
        tramo.set_attribute(ga.GEN_AI_OUTPUT_MESSAGES, respuesta)
        return respuesta, n_entrada, n_salida


def agente(consulta, max_vueltas=4):
    with tracer.start_as_current_span("invoke_agent") as raiz:
        raiz.set_attribute(ga.GEN_AI_OPERATION_NAME, Operacion.INVOKE_AGENT.value)
        raiz.set_attribute(ga.GEN_AI_AGENT_NAME, "secretaria")
        raiz.set_attribute(ga.GEN_AI_PROMPT_NAME, VERSION_PROMPT)
        # Atributos propios: el estándar no cubre a quién atender ni de quién cobrar.
        raiz.set_attribute("secretaria.alumno_id", ALUMNO)
        raiz.set_attribute("secretaria.consulta", consulta)

        mensajes = [{"role": "system", "content": SISTEMA},
                    {"role": "user", "content": consulta}]
        total_entrada = total_salida = vueltas = 0

        for _ in range(max_vueltas):
            respuesta, n_e, n_s = llamar_modelo(mensajes)
            total_entrada += n_e
            total_salida += n_s
            vueltas += 1

            encontrado = PATRON.search(respuesta)
            if not encontrado:
                raiz.set_attribute(ga.GEN_AI_USAGE_INPUT_TOKENS, total_entrada)
                raiz.set_attribute(ga.GEN_AI_USAGE_OUTPUT_TOKENS, total_salida)
                raiz.set_attribute("secretaria.vueltas", vueltas)
                raiz.set_attribute("secretaria.respuesta", respuesta)
                return respuesta

            llamada = json.loads(encontrado.group(1))
            nombre = llamada["name"]
            argumentos = llamada.get("arguments", {})

            with tracer.start_as_current_span("execute_tool") as th:
                th.set_attribute(ga.GEN_AI_OPERATION_NAME, Operacion.EXECUTE_TOOL.value)
                th.set_attribute(ga.GEN_AI_TOOL_NAME, nombre)
                th.set_attribute(ga.GEN_AI_TOOL_CALL_ARGUMENTS,
                                 json.dumps(argumentos, ensure_ascii=False))
                resultado = (CATALOGO[nombre](**argumentos) if nombre in CATALOGO
                             else f"No existe la herramienta {nombre}.")
                th.set_attribute(ga.GEN_AI_TOOL_CALL_RESULT, resultado)

            mensajes.append({"role": "assistant", "content": "",
                             "tool_calls": [{"type": "function", "function": llamada}]})
            mensajes.append({"role": "tool", "name": nombre, "content": resultado})

        raiz.set_attribute("secretaria.vueltas", vueltas)
        return "(sin respuesta: se agotaron las vueltas)"


print(agente("¿hasta cuándo puedo pedir la beca?"))

## El árbol

Los tramos que acabamos de emitir ya están en el exportador. Reconstruir el árbol es cuestión de mirar quién es padre de quién.

In [ ]:
def arbol(tramos):
    por_id = {t.context.span_id: t for t in tramos}

    def profundidad(t):
        n = 0
        while t.parent and t.parent.span_id in por_id:
            t = por_id[t.parent.span_id]
            n += 1
        return n

    print(f"{'tramo':34s} {'ms':>7s} {'in':>6s} {'out':>5s}  detalle")
    print("-" * 76)
    for t in sorted(tramos, key=lambda s: s.start_time):
        a = t.attributes
        detalle = a.get(ga.GEN_AI_TOOL_NAME) or a.get("secretaria.consulta", "") or ""
        print(f"{'  ' * profundidad(t)}{t.name:{34 - 2 * profundidad(t)}s} "
              f"{(t.end_time - t.start_time) / 1e6:7.0f} "
              f"{a.get(ga.GEN_AI_USAGE_INPUT_TOKENS, '-'):>6} "
              f"{a.get(ga.GEN_AI_USAGE_OUTPUT_TOKENS, '-'):>5}  {str(detalle)[:30]}")


arbol(memoria.get_finished_spans())

Eso es una traza. La misma forma del diagrama del capítulo, generada por el código.

Y ya se lee algo que un registro plano no diría: **dónde se va el tiempo**. La ejecución de la herramienta es una consulta a una base de datos y tarda milisegundos; el resto es el modelo. Optimizar la consulta SQL de esa herramienta sería tirar el tiempo, y sin la traza uno no lo sabe.

## Todo lo que hace falta para depurar

El capítulo insiste en registrar **el contexto exacto, no la plantilla**. Esto es lo que significa en la práctica.

In [ ]:
tramos = memoria.get_finished_spans()
chats = [t for t in tramos if t.name == "chat"]

print(f"Hubo {len(chats)} llamadas al modelo. Lo que recibió la segunda:\n")
mensajes_vistos = json.loads(chats[1].attributes[ga.GEN_AI_INPUT_MESSAGES])
for m in mensajes_vistos:
    contenido = m.get("content") or json.dumps(m.get("tool_calls", ""), ensure_ascii=False)
    print(f"  [{m['role']:9s}] {contenido[:96]}")

Ahí está el resultado de la herramienta metido en la conversación, que es justo lo que hay que ver cuando el agente responde algo raro. Sin este atributo, la pregunta "¿por qué ha dicho eso?" no tiene respuesta y se contesta con conjeturas.

Es también el atributo que más pesa y el que más problemas de privacidad trae. Volvemos a ello enseguida.

## El coste, que ahora es una dimensión

Con los tokens en los tramos, el coste deja de ser una factura a fin de mes.

Nuestro modelo es local y no cuesta dinero, así que aplicamos las tarifas de un proveedor comercial cualquiera para que los números signifiquen algo. Lo importante no son las tarifas sino que **el cálculo sale de la traza**.

In [ ]:
EUR_POR_MILLON_ENTRADA = 0.15
EUR_POR_MILLON_SALIDA = 0.60


def coste_de(tramo):
    a = tramo.attributes
    entrada = a.get(ga.GEN_AI_USAGE_INPUT_TOKENS, 0)
    salida = a.get(ga.GEN_AI_USAGE_OUTPUT_TOKENS, 0)
    return (entrada * EUR_POR_MILLON_ENTRADA + salida * EUR_POR_MILLON_SALIDA) / 1e6


raices = [t for t in tramos if t.name == "invoke_agent"]
for r in raices:
    a = r.attributes
    print(f"consulta : {a['secretaria.consulta']}")
    print(f"  vueltas: {a.get('secretaria.vueltas')}")
    print(f"  tokens : {a.get(ga.GEN_AI_USAGE_INPUT_TOKENS)} entrada / "
          f"{a.get(ga.GEN_AI_USAGE_OUTPUT_TOKENS)} salida")
    print(f"  coste  : {coste_de(r) * 1000:.4f} milésimas de euro")

### Qué determina el coste

El capítulo avisa de que el coste por tarea en un agente tiene una cola muy larga. Antes de creérnoslo, midamos de qué depende, con consultas de distinta dificultad.

In [ ]:
import statistics

memoria.clear()   # empezamos la medición limpios

CONSULTAS = [
    "¿hasta cuándo puedo pedir la beca?",
    "¿qué nota saqué en cálculo?",
    "¿cuándo se abre la matrícula extraordinaria?",
    "¿de cuántas asignaturas estoy matriculado?",
    "¿cuántas veces me puedo presentar a una asignatura?",
    "¿dónde está la cafetería?",
]

for c in CONSULTAS:
    agente(c)

raices = sorted([t for t in memoria.get_finished_spans() if t.name == "invoke_agent"],
                key=coste_de)

print(f"{'coste (milésimas)':>18s} {'vueltas':>8s}  consulta")
print("-" * 72)
for r in raices:
    print(f"{coste_de(r) * 1000:18.4f} {r.attributes.get('secretaria.vueltas', '-'):>8}  "
          f"{r.attributes['secretaria.consulta'][:40]}")

costes = [coste_de(r) for r in raices]
print(f"\nmínimo  : {min(costes) * 1000:.4f} milésimas")
print(f"mediana : {statistics.median(costes) * 1000:.4f} milésimas")
print(f"máximo  : {max(costes) * 1000:.4f} milésimas")

por_vueltas = {}
for r in raices:
    por_vueltas.setdefault(r.attributes.get("secretaria.vueltas"), []).append(coste_de(r))
print("\ncoste medio según las vueltas que dio el bucle:")
for v in sorted(por_vueltas):
    print(f"  {v} vuelta(s): {statistics.mean(por_vueltas[v]) * 1000:.4f} milésimas")

Aquí hay que ser honesto con lo que sale: **la cola larga no aparece**. Todas las consultas se resolvieron en una o dos vueltas, así que entre la más barata y la más cara hay poco más de un factor dos, y con seis casos no se puede hablar de percentiles.

Lo que sí queda medido es el mecanismo, que es lo que importa: **el coste no lo decide la consulta, lo decide cuántas vueltas dio el bucle**. Una vuelta cuesta la mitad que dos, y ninguna consulta es intrínsecamente cara.

La cola aparece cuando el bucle se atasca, y eso no lo provoca una pregunta difícil sino un fallo. Ya lo vimos en los cuadernos anteriores: la consulta de normativa que se enrutaba al expediente, la herramienta que devolvía "no estás matriculado de eso" y el agente volviendo a intentarlo. Esas son las que dan cuatro vueltas.

Y ahí está lo que hace peligrosa la media, porque el efecto se multiplica por dos vías a la vez. Cada vuelta extra arrastra todo el contexto anterior, así que el coste crece **más deprisa** que el número de vueltas. Veamos cuánto, con la advertencia de que esto es una estimación y no una medida.

In [ ]:
base = statistics.mean(por_vueltas[min(por_vueltas)])

print("proyección a partir del coste medido de una vuelta:")
print(f"{'vueltas':>8s} {'coste':>12s} {'x la de 1':>11s}")
for v in [1, 2, 4, 6, 8]:
    # cada vuelta reenvía lo acumulado: la suma crece como v(v+1)/2
    estimado = base * v * (v + 1) / 2
    print(f"{v:>8d} {estimado * 1000:>12.4f} {estimado / base:>10.1f}x")

if 2 in por_vueltas:
    medido = statistics.mean(por_vueltas[2])
    proyectado = base * 3
    print(f"\ncontraste en el único punto que sí hemos medido, 2 vueltas:")
    print(f"  medido     {medido * 1000:.4f}")
    print(f"  proyectado {proyectado * 1000:.4f}  ({proyectado / medido:.1f}x el medido)")

La última comparación es importante y por eso está puesta: **la proyección exagera**. Donde tenemos medida de verdad, dos vueltas, predice bastante más de lo que costó.

La razón es que el modelo cuadrático supone que en cada vuelta se reenvía una cantidad de contexto que crece un tramo entero, y en la práctica una parte del contexto **no crece**: el prompt de sistema y las definiciones de las herramientas son un coste fijo que se paga igual en la primera vuelta que en la octava. Lo que crece es solo el historial acumulado.

O sea que la tabla es una **cota superior**, no una predicción. Sirve para lo que sirve, que es enseñar la forma de la curva: el crecimiento no es lineal y se dispara. Para poner un presupuesto de verdad hay que medir con vueltas de verdad, no proyectar desde una.

Que es, exactamente, el argumento de todo el capítulo.

## Las trazas contienen todo

Ahora la parte incómoda. Miremos qué hemos guardado, sin filtrar.

In [ ]:
consulta_sensible = "¿qué nota saqué en cálculo?"
tramo = next(t for t in memoria.get_finished_spans()
             if t.attributes.get("secretaria.consulta") == consulta_sensible)

hijos = [t for t in memoria.get_finished_spans()
         if t.parent and t.parent.span_id == tramo.context.span_id]

print("En los tramos de esa consulta hay guardado:")
print(f"  identificador del alumno : {tramo.attributes.get('secretaria.alumno_id')}")
for h in hijos:
    if h.name == "execute_tool":
        print(f"  resultado de {h.attributes[ga.GEN_AI_TOOL_NAME]}: "
              f"{h.attributes[ga.GEN_AI_TOOL_CALL_RESULT][:80]}")

Su expediente entero, con nombre de asignatura y nota, en un atributo de una traza.

El capítulo lo dice sin rodeos: **vuestro sistema de observabilidad hereda la clasificación del dato más sensible que pase por el sistema**. Si esas trazas se van a un SaaS, acabáis de mandar expedientes académicos a un tercero, y probablemente sin que nadie lo haya revisado, porque la conversación sobre proveedores se tuvo con el del modelo y no con el del visor de trazas.

Y recordad el [artículo 21 de la normativa](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/rag.html) que indexamos: el expediente es información personal de su titular. Un visor de trazas al que tenga acceso todo el equipo de desarrollo es, exactamente, el canal que ese artículo prohíbe.

La corrección se aplica **antes de guardar**, no después.

In [ ]:
import re as _re

PATRONES = [
    (_re.compile(r"\bA\d{7}\b"), "[ALUMNO]"),
    (_re.compile(r"\b\d{1,2}\.\d\b"), "[NOTA]"),
    (_re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "[EMAIL]"),
]


def enmascarar(texto):
    for patron, reemplazo in PATRONES:
        texto = patron.sub(reemplazo, texto)
    return texto


ejemplo = (f"El alumno {ALUMNO} pregunta. Expediente: "
           f"{consultar_expediente()}. Contacto: aitor.agirre0@alumnos.uni-ficticia.es")

print("SIN ENMASCARAR:")
print(f"  {ejemplo[:150]}")
print("\nCOMO DEBERÍA GUARDARSE:")
print(f"  {enmascarar(ejemplo)[:150]}")

Es un enmascarado ingenuo y conviene decirlo: se le escapará cualquier cosa que no encaje en sus tres expresiones regulares, y el nombre y apellidos del alumno pasan enteros. Sirve para enseñar dónde va la pieza, no como implementación.

Lo que sí es transferible es **dónde** se coloca: entre el dato y el `set_attribute`, dentro de vuestro proceso. No en el visor, no en una política de la herramienta, no en un acuerdo con el proveedor. Si el dato sale del proceso sin enmascarar, ya salió.

Junto con eso, las otras tres decisiones que el capítulo pide tomar antes de instrumentar y no después: **retención** con un plazo escrito, **control de acceso** al visor y, si es SaaS, **la misma revisión de proveedor** que se le hizo al modelo.

## Del registro a la mejora

Queda el paso que convierte todo esto en algo más que un archivo de incidentes. El capítulo lo plantea como un bucle: las trazas revelan fallos reales, esos casos entran en el conjunto de evaluación, y los cambios se validan contra él.

Lo bueno es que una traza ya tiene la forma de un caso de prueba. Solo hay que sacarla.

In [ ]:
def a_caso_de_evaluacion(raiz, tramos):
    """Convierte una traza en un caso, listo para revisar y etiquetar."""
    hijos = [t for t in tramos if t.parent and t.parent.span_id == raiz.context.span_id]
    herramientas = [t.attributes[ga.GEN_AI_TOOL_NAME]
                    for t in hijos if t.name == "execute_tool"]
    return {
        "consulta": raiz.attributes["secretaria.consulta"],
        "respuesta_obtenida": raiz.attributes.get("secretaria.respuesta", ""),
        "herramientas_usadas": herramientas,
        "vueltas": raiz.attributes.get("secretaria.vueltas"),
        "version_prompt": raiz.attributes.get(ga.GEN_AI_PROMPT_NAME),
        "respuesta_esperada": None,   # esto lo rellena una persona
    }


todos = memoria.get_finished_spans()
casos = [a_caso_de_evaluacion(r, todos) for r in todos if r.name == "invoke_agent"]

print(json.dumps(casos[:2], ensure_ascii=False, indent=2))
print(f"\n{len(casos)} casos extraídos de las trazas.")

Fijaos en el campo `respuesta_esperada`, que va vacío a propósito. Ese hueco es todo el trabajo: **una traza dice lo que pasó, no si estuvo bien**. Quien decide eso es una persona, mirando la consulta y la respuesta.

Y también en `version_prompt`. Sin él, cuando el sistema empeore no habrá forma de atribuirlo a un cambio concreto, que es la diferencia entre depurar y adivinar.

Esos casos, ya etiquetados, son la entrada del [cuaderno de evaluación](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/evaluacion.html). Y ahí está lo que el capítulo llama la única ventaja acumulativa que se puede construir en esto: los modelos los tiene todo el mundo, vuestro conjunto de casos reales no.

## Ejercicios

**1. El tramo que falta.** El agente de los cuadernos de contexto hacía recuperación. Añadid un tramo `retrieval` con `gen_ai.retrieval.query.text` y los documentos devueltos, y mirad qué se ve al fallar una respuesta: si el problema fue traer mal o redactar mal.

**2. Errores que se ven.** Provocad un fallo en una herramienta y comprobad qué queda en la traza. Después usad `tramo.record_exception()` y `set_status()` y comparad. Un tramo que no marca su estado es un fallo invisible en cualquier panel.

**3. Atribuir gasto.** Añadid un atributo de sesión y otro de caso de uso, lanzad consultas de varios alumnos y agregad el coste por alumno y por tipo de consulta. Eso es lo que hace posible una conversación sobre presupuesto.

**4. El tope duro.** Con el coste ya medido por traza, poned un presupuesto por tarea que corte el bucle al superarse, como en el cuaderno del bucle, y emitid un tramo que lo registre. Comprobad que se ve en el árbol.

**5. Enmascarar de verdad.** Nuestro enmascarado deja pasar nombres y apellidos. Ampliadlo con los del almacén y medid cuántos se le escapan sobre las trazas ya recogidas. Es un buen recordatorio de lo difícil que es esto.

**6. Cambiar de exportador.** Sustituid `InMemorySpanExporter` por `ConsoleSpanExporter` y luego por uno OTLP. Contad las líneas de instrumentación que hay que tocar. Si la respuesta es cero, habéis entendido para qué sirve el estándar.

## Lo que os lleváis

* **La traza es la unidad**, no la línea de registro. Un árbol de tramos enseña dónde se va el tiempo y por qué se dijo lo que se dijo.
* **Los nombres ya están decididos.** Las convenciones de IA generativa cubren casi toda la lista del capítulo, y lo que no cubren son vuestros identificadores.
* **La latencia sale gratis.** Un tramo sabe cuándo empezó y cuándo acabó.
* **Guardad el contexto exacto**, no la plantilla. Es lo que convierte "¿por qué ha dicho eso?" en una pregunta con respuesta.
* **El coste lo deciden las vueltas del bucle**, no la consulta, y crece con el cuadrado de las vueltas. Mirad la distribución, no la media.
* **Los fallos son lo caro.** Las tareas donde el agente se lía son a la vez las peor respondidas y las que se llevan el presupuesto.
* **La traza lleva dentro el dato más sensible del sistema.** Se enmascara antes del `set_attribute`, dentro de vuestro proceso.
* **Una traza no dice si estuvo bien.** Dice qué pasó. Lo otro lo pone una persona, y eso es [evaluar](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/evaluacion.html).